In [ ]:
import platform
import subprocess
import sys


if platform.system() == "Windows":
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "torch==2.14.0+cpu",
            "--index-url",
            "https://download.pytorch.org/whl/cpu",
        ]
    )

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sentence-transformers",
        "datasets",
        "pandas",
        "numpy",
        "scikit-learn",
    ]
)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 01 - Experiment Objective

Determine which embedding model gives this production RAG system the best retrieval quality, latency, and resource profile. The initial benchmark uses English enterprise-style content; multilingual English/Urdu, Arabic, and other enterprise content should be added before final production selection.

# 02 - Environment Setup

Colab can use a GPU runtime for model inference. The benchmark records the selected device and keeps model downloads outside the laptop environment.

In [3]:
import time

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print(f"NumPy: {np.__version__}")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Projects\enterprise-ai-knowledge-assistant\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

# 03 - Dataset Preparation

The initial benchmark is intentionally small and English-only. A serious evaluation should expand this with representative enterprise documents, queries, and multilingual coverage.

In [ ]:
documents = [
    "Employees receive twenty four annual leave days.",
    "Employees must submit leave requests through the HR portal.",
    "Medical leave requires supporting medical documentation.",
    "The IT department manages employee laptop provisioning.",
    "Password resets must be requested through the service desk.",
    "Managers approve annual leave requests.",
]

queries = [
    "How many annual leave days do employees receive?",
    "How do employees submit leave requests?",
    "What is required for medical leave?",
    "Who manages employee laptops?",
    "How do I reset my password?",
    "Who approves annual leave?",
]

relevant_documents = {
    0: [0],
    1: [1],
    2: [2],
    3: [3],
    4: [4],
    5: [5],
}

benchmark = pd.DataFrame({"query": queries})
benchmark

# 04 - Candidate Embedding Models

The initial candidates cover a compact general-purpose model and two BGE models. Popularity is not a selection criterion; the benchmark decides.

In [ ]:
MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
]

MODELS

# 05 - Embedding Generation

Start with `all-MiniLM-L6-v2`; later loop over `MODELS` and store each model's metrics.

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Device:", model.device)

document_embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True,
)
query_embeddings = model.encode(
    queries,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Document embeddings:", document_embeddings.shape)
print("Query embeddings:", query_embeddings.shape)

# 06 - Retrieval Benchmark

Normalized embeddings make the dot product equivalent to cosine similarity for this benchmark.

In [ ]:
similarities = query_embeddings @ document_embeddings.T

print("Similarity matrix:", similarities.shape)


def retrieve(
    query_index: int,
    k: int = 5,
):
    scores = similarities[query_index]
    ranked_indices = np.argsort(scores)[::-1]
    return ranked_indices[:k]


retrieve(0, k=3)

# 07 - Recall@K

In [ ]:
def recall_at_k(
    similarities,
    ground_truth,
    k,
):
    hits = 0

    for query_index, relevant in ground_truth.items():
        ranked = np.argsort(similarities[query_index])[::-1][:k]

        if any(index in relevant for index in ranked):
            hits += 1

    return hits / len(ground_truth)


for k in [1, 3, 5]:
    print(
        f"Recall@{k}:",
        recall_at_k(
            similarities,
            relevant_documents,
            k,
        ),
    )

# 08 - MRR

In [ ]:
def mean_reciprocal_rank(
    similarities,
    ground_truth,
):
    reciprocal_ranks = []

    for query_index, relevant in ground_truth.items():
        ranked = np.argsort(similarities[query_index])[::-1]

        reciprocal_rank = 0.0

        for rank, index in enumerate(
            ranked,
            start=1,
        ):
            if index in relevant:
                reciprocal_rank = 1 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

    return np.mean(reciprocal_ranks)


mrr = mean_reciprocal_rank(
    similarities,
    relevant_documents,
)

print("MRR:", mrr)

# 09 - Latency

In [ ]:
def measure_query_latency(model, query_batch, runs=5):
    latencies = []

    for _ in range(runs):
        start = time.perf_counter()
        model.encode(query_batch, normalize_embeddings=True, show_progress_bar=False)
        latencies.append(time.perf_counter() - start)

    return {
        "mean_seconds": float(np.mean(latencies)),
        "p95_seconds": float(np.percentile(latencies, 95)),
    }


latency = measure_query_latency(model, queries)
latency

# 10 - Memory

In [ ]:
memory = {
    "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
    "embedding_dimension": int(document_embeddings.shape[1]),
    "device": str(model.device),
}

pd.Series(memory)

# 11 - Results

In [ ]:
results = []

for model_name in MODELS:
    print(f"Testing {model_name}")

    start = time.perf_counter()

    model = SentenceTransformer(model_name)

    document_embeddings = model.encode(
        documents,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    query_embeddings = model.encode(
        queries,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    elapsed = time.perf_counter() - start

    similarities = query_embeddings @ document_embeddings.T

    results.append(
        {
            "model": model_name,
            "recall_at_1": recall_at_k(
                similarities,
                relevant_documents,
                1,
            ),
            "recall_at_3": recall_at_k(
                similarities,
                relevant_documents,
                3,
            ),
            "recall_at_5": recall_at_k(
                similarities,
                relevant_documents,
                5,
            ),
            "mrr": mean_reciprocal_rank(
                similarities,
                relevant_documents,
            ),
            "experiment_seconds": elapsed,
            "embedding_dimension": document_embeddings.shape[1],
        }
    )

results_df = pd.DataFrame(results)
results_df

## Prototype Limitations

This six-question benchmark establishes the methodology only. The final evaluation should include 500+ question/document pairs across HR, IT, Finance, Procurement, Operations, Policies, Compliance, and technical documentation.

## Engineering Decision Criteria

The production choice should balance retrieval quality, latency, embedding dimension, GPU/CPU requirements, cost, deployment complexity, and language coverage. Multilingual English/Urdu, Arabic, and other enterprise content should be benchmarked before final selection.

# 12 - Model Selection

Select the production candidate from measured retrieval quality first, then use latency and memory as constraints. Do not select by model popularity.

In [ ]:
selected_model = results_df.sort_values(
    ["recall_at_5", "mrr", "experiment_seconds"],
    ascending=[False, False, True],
).iloc[0]

print("Selected model:", selected_model["model"])
selected_model

# 13 - Export Configuration

Export the measured winner into the application configuration only after the larger, representative benchmark and multilingual evaluation are complete.

In [ ]:
export_config = {
    "EMBEDDING_PROVIDER": "sentence_transformers",
    "EMBEDDING_MODEL": str(selected_model["model"]),
    "EMBEDDING_DIMENSION": int(selected_model["embedding_dimension"]),
}

export_config